## CELL 1 — Cài thư viện
Cài đủ thư viện cho:
- xử lý parquet lớn bằng **DuckDB**
- thao tác dữ liệu bằng **pandas / pyarrow**
- tạo feature matrix và chạy **Truncated SVD** bằng **scikit-learn**

In [1]:
!pip -q install duckdb pyarrow pandas fastparquet scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.1 MB/s eta 0:00:00


## CELL 2 — Khởi tạo thư mục, cấu hình dataset, tải dữ liệu gốc

Cell này:
1. Tạo cấu trúc thư mục version hóa.
2. Tải đủ 12 file parquet năm 2024.
3. Chỉ lưu vào thư mục **raw**, tuyệt đối không xử lý đè lên dữ liệu gốc.

In [2]:
import os
import urllib.request
from pathlib import Path

# =========================
# 1. CẤU HÌNH DATASET
# =========================
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATASET_PREFIX = "yellow_tripdata"
YEAR = 2024
MONTHS = list(range(1, 13))  # tải đủ 12 tháng

# =========================
# 2. CẤU TRÚC THƯ MỤC VERSION
# =========================
PROJECT_DIR = Path("/content/project_bigdata_svd")
RAW_DIR = PROJECT_DIR / "data" / "raw"
V1_DIR = PROJECT_DIR / "data" / "v1_loaded"
V2_DIR = PROJECT_DIR / "data" / "v2_cleaned_nulls"
V3_DIR = PROJECT_DIR / "data" / "v3_deduplicated"
V4_DIR = PROJECT_DIR / "data" / "v4_validated"
V5_DIR = PROJECT_DIR / "data" / "v5_normalized"
V6_DIR = PROJECT_DIR / "data" / "v6_svd_input"
V7_DIR = PROJECT_DIR / "data" / "v7_svd_output"
FINAL_DIR = PROJECT_DIR / "data" / "final"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [RAW_DIR, V1_DIR, V2_DIR, V3_DIR, V4_DIR, V5_DIR, V6_DIR, V7_DIR, FINAL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

# =========================
# 3. TẢI DỮ LIỆU GỐC
# =========================
downloaded_files = []

for month in MONTHS:
    mm = f"{month:02d}"
    filename = f"{DATASET_PREFIX}_{YEAR}-{mm}.parquet"
    url = f"{BASE_URL}/{filename}"
    target_path = RAW_DIR / filename

    if target_path.exists():
        print(f"[Skip] {filename} đã tồn tại")
    else:
        print(f"[Download] {filename}")
        urllib.request.urlretrieve(url, target_path)

    downloaded_files.append(str(target_path))

print("\nĐã sẵn sàng dữ liệu gốc:")
for f in downloaded_files[:3]:
    print(" -", f)
print(" ...")
print("Tổng số file:", len(downloaded_files))

Project directory: /content/project_bigdata_svd
[Download] yellow_tripdata_2024-01.parquet
[Download] yellow_tripdata_2024-02.parquet
[Download] yellow_tripdata_2024-03.parquet
[Download] yellow_tripdata_2024-04.parquet
[Download] yellow_tripdata_2024-05.parquet
[Download] yellow_tripdata_2024-06.parquet
[Download] yellow_tripdata_2024-07.parquet
[Download] yellow_tripdata_2024-08.parquet
[Download] yellow_tripdata_2024-09.parquet
[Download] yellow_tripdata_2024-10.parquet
[Download] yellow_tripdata_2024-11.parquet
[Download] yellow_tripdata_2024-12.parquet

Đã sẵn sàng dữ liệu gốc:
 - /content/project_bigdata_svd/data/raw/yellow_tripdata_2024-01.parquet
 - /content/project_bigdata_svd/data/raw/yellow_tripdata_2024-02.parquet
 - /content/project_bigdata_svd/data/raw/yellow_tripdata_2024-03.parquet
 ...
Tổng số file: 12


## CELL 3 — Đọc schema, xem mẫu dữ liệu, lưu **Version 1**

**Version 1** chỉ là trạng thái “đã load thành công”, chưa chỉnh sửa dữ liệu.

In [3]:
import duckdb
import pandas as pd

con = duckdb.connect()
raw_glob = str(RAW_DIR / "*.parquet")
v1_file = str(V1_DIR / "v1_loaded.parquet")

# 1. Xem schema toàn bộ dữ liệu gốc
schema_df = con.execute(f'''
DESCRIBE SELECT * FROM read_parquet('{raw_glob}')
''').fetchdf()

print("=== SCHEMA OF RAW DATA ===")
display(schema_df)

# 2. Xem mẫu dữ liệu
sample_df = con.execute(f'''
SELECT * FROM read_parquet('{raw_glob}')
LIMIT 10
''').fetchdf()

print("=== SAMPLE RAW DATA ===")
display(sample_df)

# 3. Đếm tổng số dòng
raw_count_df = con.execute(f'''
SELECT COUNT(*) AS raw_total_rows
FROM read_parquet('{raw_glob}')
''').fetchdf()

print("=== RAW TOTAL ROWS ===")
display(raw_count_df)

# 4. Lưu version 1
con.execute(f'''
COPY (
    SELECT *
    FROM read_parquet('{raw_glob}')
) TO '{v1_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

print("Saved V1:", v1_file)

=== SCHEMA OF RAW DATA ===


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


=== SAMPLE RAW DATA ===


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.00
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.00
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.00
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.00
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.00
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.70,1,N,148,141,1,29.6,3.5,0.5,6.90,0.0,1.0,41.50,2.5,0.00
6,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.00,0.0,1.0,64.95,0.0,1.75
7,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.00,1,N,246,231,2,25.4,3.5,0.5,0.00,0.0,1.0,30.40,2.5,0.00
8,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.00,0.0,1.0,36.00,2.5,0.00
9,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.00,0.0,1.0,8.00,2.5,0.00


=== RAW TOTAL ROWS ===


,raw_total_rows
0,41169720


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V1: /content/project_bigdata_svd/data/v1_loaded/v1_loaded.parquet


## CELL 4 — Task 1 / Data Cleaning: kiểm tra NULL, xử lý giá trị thiếu, lưu **Version 2**

Nguyên tắc:
- cột thời gian và các cột lõi của chuyến đi: nếu thiếu thì loại
- một số cột số có thể fill hợp lý nếu cần
- mọi xử lý đều được lưu thành version mới

In [4]:
v1_file = str(V1_DIR / "v1_loaded.parquet")
v2_file = str(V2_DIR / "v2_cleaned_nulls.parquet")

# 1. Báo cáo null ở các cột quan trọng
null_report_df = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS VendorID_nulls,
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_datetime_nulls,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS dropoff_datetime_nulls,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_nulls,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS PULocationID_nulls,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS DOLocationID_nulls,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_nulls,
    SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) AS RatecodeID_nulls,
    SUM(CASE WHEN store_and_fwd_flag IS NULL THEN 1 ELSE 0 END) AS store_and_fwd_flag_nulls
FROM read_parquet('{v1_file}')
''').fetchdf()

print("=== NULL REPORT BEFORE ===")
display(null_report_df)

null_report_df.to_csv(REPORT_DIR / "null_report_before.csv", index=False)

# 2. Xử lý null
# - Loại các dòng thiếu dữ liệu lõi
# - Fill store_and_fwd_flag = 'N' khi thiếu
# - Fill passenger_count = 1 khi thiếu
# - Fill RatecodeID = 1 khi thiếu
con.execute(f'''
COPY (
    SELECT
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        COALESCE(passenger_count, 1) AS passenger_count,
        trip_distance,
        COALESCE(RatecodeID, 1) AS RatecodeID,
        COALESCE(store_and_fwd_flag, 'N') AS store_and_fwd_flag,
        PULocationID,
        DOLocationID,
        payment_type,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge,
        Airport_fee
    FROM read_parquet('{v1_file}')
    WHERE VendorID IS NOT NULL
      AND tpep_pickup_datetime IS NOT NULL
      AND tpep_dropoff_datetime IS NOT NULL
      AND trip_distance IS NOT NULL
      AND PULocationID IS NOT NULL
      AND DOLocationID IS NOT NULL
      AND payment_type IS NOT NULL
      AND total_amount IS NOT NULL
) TO '{v2_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

print("Saved V2:", v2_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== NULL REPORT BEFORE ===


,total_rows,VendorID_nulls,pickup_datetime_nulls,dropoff_datetime_nulls,passenger_count_nulls,trip_distance_nulls,PULocationID_nulls,DOLocationID_nulls,payment_type_nulls,total_amount_nulls,RatecodeID_nulls,store_and_fwd_flag_nulls
0,41169720,0.0,0.0,0.0,4091232.0,0.0,0.0,0.0,0.0,0.0,4091232.0,4091232.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V2: /content/project_bigdata_svd/data/v2_cleaned_nulls/v2_cleaned_nulls.parquet


## CELL 5 — Task 1 / Data Cleaning: xóa duplicate, lưu **Version 3**

In [5]:
v2_file = str(V2_DIR / "v2_cleaned_nulls.parquet")
v3_file = str(V3_DIR / "v3_deduplicated.parquet")

dup_report_before = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        PULocationID,
        DOLocationID,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v2_file}')
''').fetchdf()

print("=== DUPLICATE REPORT BEFORE ===")
display(dup_report_before)
dup_report_before.to_csv(REPORT_DIR / "duplicate_report_before.csv", index=False)

con.execute(f'''
COPY (
    SELECT DISTINCT *
    FROM read_parquet('{v2_file}')
) TO '{v3_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

dup_report_after = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        PULocationID,
        DOLocationID,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v3_file}')
''').fetchdf()

print("=== DUPLICATE REPORT AFTER ===")
display(dup_report_after)
dup_report_after.to_csv(REPORT_DIR / "duplicate_report_after.csv", index=False)

print("Saved V3:", v3_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DUPLICATE REPORT BEFORE ===


,total_rows,duplicate_rows
0,41169720,5


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DUPLICATE REPORT AFTER ===


,total_rows,duplicate_rows
0,41169716,1


Saved V3: /content/project_bigdata_svd/data/v3_deduplicated/v3_deduplicated.parquet


## CELL 6 — Task 1 / Data Cleaning: loại dữ liệu sai, giá trị âm bất hợp lý, lỗi thời gian, lưu **Version 4**

Ở bước này ta giữ lại các bản ghi hợp lệ để đầu vào cho SVD không bị méo vì dữ liệu lỗi.

In [6]:
v3_file = str(V3_DIR / "v3_deduplicated.parquet")
v4_file = str(V4_DIR / "v4_validated.parquet")

invalid_before_df = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN CAST(tpep_dropoff_datetime AS TIMESTAMP) < CAST(tpep_pickup_datetime AS TIMESTAMP) THEN 1 ELSE 0 END) AS bad_time_rows,
    SUM(CASE WHEN COALESCE(passenger_count, 0) < 0 THEN 1 ELSE 0 END) AS negative_passenger_rows,
    SUM(CASE WHEN COALESCE(trip_distance, 0) < 0 THEN 1 ELSE 0 END) AS negative_trip_distance_rows,
    SUM(CASE WHEN COALESCE(fare_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_fare_rows,
    SUM(CASE WHEN COALESCE(extra, 0) < 0 THEN 1 ELSE 0 END) AS negative_extra_rows,
    SUM(CASE WHEN COALESCE(mta_tax, 0) < 0 THEN 1 ELSE 0 END) AS negative_mta_tax_rows,
    SUM(CASE WHEN COALESCE(tip_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_tip_rows,
    SUM(CASE WHEN COALESCE(tolls_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_tolls_rows,
    SUM(CASE WHEN COALESCE(improvement_surcharge, 0) < 0 THEN 1 ELSE 0 END) AS negative_improvement_rows,
    SUM(CASE WHEN COALESCE(total_amount, 0) < -100 THEN 1 ELSE 0 END) AS extreme_negative_total_rows
FROM read_parquet('{v3_file}')
''').fetchdf()

print("=== INVALID DATA REPORT BEFORE ===")
display(invalid_before_df)
invalid_before_df.to_csv(REPORT_DIR / "invalid_report_before.csv", index=False)

# Điều kiện lọc:
# - thời gian dropoff >= pickup
# - khoảng cách, tiền cơ bản không âm
# - passenger_count hợp lệ
con.execute(f'''
COPY (
    SELECT *
    FROM read_parquet('{v3_file}')
    WHERE CAST(tpep_dropoff_datetime AS TIMESTAMP) >= CAST(tpep_pickup_datetime AS TIMESTAMP)
      AND COALESCE(passenger_count, 0) >= 0
      AND COALESCE(trip_distance, 0) >= 0
      AND COALESCE(fare_amount, 0) >= 0
      AND COALESCE(extra, 0) >= 0
      AND COALESCE(mta_tax, 0) >= 0
      AND COALESCE(tolls_amount, 0) >= 0
      AND COALESCE(improvement_surcharge, 0) >= 0
      AND COALESCE(total_amount, 0) > -100
) TO '{v4_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

print("Saved V4:", v4_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== INVALID DATA REPORT BEFORE ===


,total_rows,bad_time_rows,negative_passenger_rows,negative_trip_distance_rows,negative_fare_rows,negative_extra_rows,negative_mta_tax_rows,negative_tip_rows,negative_tolls_rows,negative_improvement_rows,extreme_negative_total_rows
0,41169716,1575.0,0.0,0.0,731024.0,304569.0,583164.0,1331.0,49272.0,606112.0,9704.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V4: /content/project_bigdata_svd/data/v4_validated/v4_validated.parquet


## CELL 7 — Task 1 / Data Normalization & Feature Engineering: chuẩn hoá kiểu dữ liệu, format, text, lưu **Version 5**

Bước này tạo ra dữ liệu **consistent** và sẵn sàng cho việc tạo ma trận đặc trưng ở Task

In [7]:
v4_file = str(V4_DIR / "v4_validated.parquet")
v5_file = str(V5_DIR / "v5_normalized.parquet")

con.execute(f'''
COPY (
    SELECT
        CAST(VendorID AS BIGINT) AS vendor_id,
        CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) AS dropoff_datetime,

        CAST(passenger_count AS BIGINT) AS passenger_count,
        CAST(trip_distance AS DOUBLE) AS trip_distance,
        CAST(RatecodeID AS BIGINT) AS rate_code_id,

        CASE
            WHEN UPPER(TRIM(CAST(store_and_fwd_flag AS VARCHAR))) IN ('Y', 'N')
                THEN UPPER(TRIM(CAST(store_and_fwd_flag AS VARCHAR)))
            ELSE 'N'
        END AS store_and_fwd_flag,

        CAST(PULocationID AS BIGINT) AS pu_location_id,
        CAST(DOLocationID AS BIGINT) AS do_location_id,
        CAST(payment_type AS BIGINT) AS payment_type,

        ROUND(CAST(fare_amount AS DOUBLE), 2) AS fare_amount,
        ROUND(CAST(extra AS DOUBLE), 2) AS extra,
        ROUND(CAST(mta_tax AS DOUBLE), 2) AS mta_tax,
        ROUND(CAST(tip_amount AS DOUBLE), 2) AS tip_amount,
        ROUND(CAST(tolls_amount AS DOUBLE), 2) AS tolls_amount,
        ROUND(CAST(improvement_surcharge AS DOUBLE), 2) AS improvement_surcharge,
        ROUND(CAST(total_amount AS DOUBLE), 2) AS total_amount,
        ROUND(CAST(congestion_surcharge AS DOUBLE), 2) AS congestion_surcharge,
        ROUND(CAST(Airport_fee AS DOUBLE), 2) AS airport_fee,

        -- Dẫn xuất thời gian phục vụ phân tích và SVD
        EXTRACT(YEAR FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_year,
        EXTRACT(MONTH FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_month,
        EXTRACT(DAY FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_day,
        EXTRACT(HOUR FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_hour,
        EXTRACT(DOW FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_dow,
        DATE_DIFF('minute',
            CAST(tpep_pickup_datetime AS TIMESTAMP),
            CAST(tpep_dropoff_datetime AS TIMESTAMP)
        ) AS trip_duration_min
    FROM read_parquet('{v4_file}')
) TO '{v5_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

print("Saved V5:", v5_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V5: /content/project_bigdata_svd/data/v5_normalized/v5_normalized.parquet


## CELL 8 — Task 1 / Kiểm tra cuối và xuất dataset đã làm sạch

Mục tiêu của Task 1:
- không sửa dữ liệu gốc
- mỗi bước có version riêng
- dataset cuối **consistent**
- sẵn sàng tạo feature matrix cho SVD

In [8]:
v5_file = str(V5_DIR / "v5_normalized.parquet")
task1_parquet = str(FINAL_DIR / "task1_cleaned_normalized_yellow_taxi_2024.parquet")
task1_csv = str(FINAL_DIR / "task1_cleaned_normalized_yellow_taxi_2024.csv")

final_null_check_df = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN vendor_id IS NULL THEN 1 ELSE 0 END) AS vendor_id_nulls,
    SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_datetime_nulls,
    SUM(CASE WHEN dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS dropoff_datetime_nulls,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_nulls,
    SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) AS pu_location_id_nulls,
    SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END) AS do_location_id_nulls,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_nulls,
    SUM(CASE WHEN trip_duration_min IS NULL THEN 1 ELSE 0 END) AS trip_duration_min_nulls
FROM read_parquet('{v5_file}')
''').fetchdf()

print("=== FINAL NULL CHECK ===")
display(final_null_check_df)
final_null_check_df.to_csv(REPORT_DIR / "final_null_check.csv", index=False)

final_dup_check_df = con.execute(f'''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        vendor_id,
        pickup_datetime,
        dropoff_datetime,
        passenger_count,
        trip_distance,
        pu_location_id,
        do_location_id,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v5_file}')
''').fetchdf()

print("=== FINAL DUPLICATE CHECK ===")
display(final_dup_check_df)
final_dup_check_df.to_csv(REPORT_DIR / "final_duplicate_check.csv", index=False)

# Xuất dữ liệu Task 1
con.execute(f'''
COPY (
    SELECT *
    FROM read_parquet('{v5_file}')
) TO '{task1_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

con.execute(f'''
COPY (
    SELECT *
    FROM read_parquet('{v5_file}')
) TO '{task1_csv}' (HEADER, DELIMITER ',')
''')

print("Task 1 Parquet:", task1_parquet)
print("Task 1 CSV:", task1_csv)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL NULL CHECK ===


,total_rows,vendor_id_nulls,pickup_datetime_nulls,dropoff_datetime_nulls,passenger_count_nulls,trip_distance_nulls,pu_location_id_nulls,do_location_id_nulls,payment_type_nulls,total_amount_nulls,trip_duration_min_nulls
0,40434154,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL DUPLICATE CHECK ===


,total_rows,duplicate_rows
0,40434154,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Task 1 Parquet: /content/project_bigdata_svd/data/final/task1_cleaned_normalized_yellow_taxi_2024.parquet
Task 1 CSV: /content/project_bigdata_svd/data/final/task1_cleaned_normalized_yellow_taxi_2024.csv


## CELL 9 — Chuẩn bị dữ liệu đầu vào cho SVD, lưu **Version 6**

Để tránh Colab hết RAM, cell này cho phép lấy **sample có kiểm soát** từ dataset rất lớn.  
Bạn vẫn dùng full dataset cho Task 1, còn Task 2 có thể:
- chạy sample khi máy yếu
- tăng sample lên khi máy mạnh

In [9]:
v5_file = str(V5_DIR / "v5_normalized.parquet")
v6_file = str(V6_DIR / "v6_svd_input.parquet")

# =========================
# CẤU HÌNH KÍCH THƯỚC MẪU
# =========================
# Có thể tăng lên nếu RAM đủ
SVD_SAMPLE_ROWS = 2_000_000

# Chọn feature phục vụ SVD
con.execute(f'''
COPY (
    SELECT
        vendor_id,
        passenger_count,
        trip_distance,
        rate_code_id,
        store_and_fwd_flag,
        pu_location_id,
        do_location_id,
        payment_type,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge,
        airport_fee,
        pickup_month,
        pickup_day,
        pickup_hour,
        pickup_dow,
        trip_duration_min
    FROM read_parquet('{v5_file}')
    USING SAMPLE {SVD_SAMPLE_ROWS} ROWS
) TO '{v6_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
''')

svd_input_count = con.execute(f'''
SELECT COUNT(*) AS svd_input_rows
FROM read_parquet('{v6_file}')
''').fetchdf()

print("=== SVD INPUT ROWS ===")
display(svd_input_count)
print("Saved V6:", v6_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== SVD INPUT ROWS ===


,svd_input_rows
0,2000000


Saved V6: /content/project_bigdata_svd/data/v6_svd_input/v6_svd_input.parquet


## CELL 10 — Tạo feature matrix cho SVD

- Numeric features → **StandardScaler**
- Categorical features → **OneHotEncoder**
- Kết quả là một **sparse matrix** phù hợp để chạy Truncated SVD

In [10]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD

# Đọc sample đầu vào từ version 6
svd_df = con.execute(f'''
SELECT *
FROM read_parquet('{v6_file}')
''').fetchdf()

print("Shape dữ liệu đầu vào cho SVD:", svd_df.shape)
display(svd_df.head())

# Kiểm tra giá trị thiếu trước khi tạo feature matrix
null_summary = svd_df.isna().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

print("=== CÁC CỘT CÒN THIẾU DỮ LIỆU TRƯỚC SVD ===")
if len(null_summary) == 0:
    print("Không còn NaN.")
else:
    display(null_summary.to_frame("null_count"))

# Chia nhóm cột
numeric_features = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "trip_duration_min"
]

categorical_features = [
    "vendor_id",
    "rate_code_id",
    "store_and_fwd_flag",
    "pu_location_id",
    "do_location_id",
    "payment_type",
    "pickup_month",
    "pickup_day",
    "pickup_hour",
    "pickup_dow"
]

# Pipeline tiền xử lý có xử lý NaN
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

X = preprocessor.fit_transform(svd_df)

print("Feature matrix type:", type(X))
print("Feature matrix shape:", X.shape)

# Kiểm tra lại sau transform
if hasattr(X, "data"):  # sparse matrix
    print("NaN trong sparse data sau preprocessing:", np.isnan(X.data).sum())
else:
    print("NaN sau preprocessing:", np.isnan(X).sum())


Shape dữ liệu đầu vào cho SVD: (2000000, 22)


,vendor_id,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_month,pickup_day,pickup_hour,pickup_dow,trip_duration_min
0,2,2,18.25,2,N,132,75,1,70.00,0.0,...,6.94,1.0,95.88,0.0,1.75,11,27,6,3,59
1,1,1,1.80,99,N,130,215,1,18.50,0.0,...,0.00,1.0,20.00,0.0,0.00,2,4,11,0,18
2,2,1,1.24,1,N,170,90,0,9.03,0.0,...,0.00,1.0,13.03,NaN,NaN,6,22,0,6,8
3,2,1,0.69,1,N,239,142,1,6.50,0.0,...,0.00,1.0,13.12,2.5,0.00,8,30,11,5,5
4,1,3,2.70,1,N,143,236,1,13.50,3.5,...,0.00,1.0,21.28,2.5,0.00,4,19,20,5,9


=== CÁC CỘT CÒN THIẾU DỮ LIỆU TRƯỚC SVD ===


,null_count
congestion_surcharge,194284
airport_fee,194284


Feature matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Feature matrix shape: (2000000, 623)
NaN trong sparse data sau preprocessing: 0


## CELL 11 — Chạy Truncated SVD, đánh giá explained variance, lưu **Version 7**

Bạn có thể đổi số chiều `N_COMPONENTS` để so sánh chất lượng nén dữ liệu.

In [11]:
from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 20

svd_model = TruncatedSVD(
    n_components=N_COMPONENTS,
    n_iter=10,
    random_state=42
)

X_reduced = svd_model.fit_transform(X)

explained_variance_ratio = svd_model.explained_variance_ratio_
explained_variance_sum = explained_variance_ratio.sum()

print("=== TRUNCATED SVD RESULT ===")
print("Original matrix shape :", X.shape)
print("Reduced matrix shape  :", X_reduced.shape)
print("Explained variance ratio (first components):")
print(explained_variance_ratio)
print("Total explained variance:", explained_variance_sum)

# Lưu kết quả giảm chiều ra DataFrame
reduced_columns = [f"svd_component_{i+1}" for i in range(N_COMPONENTS)]
reduced_df = pd.DataFrame(X_reduced, columns=reduced_columns)

# Ghép thêm vài cột gốc để tiện giải thích / phân tích sau
result_df = pd.concat(
    [
        svd_df[["total_amount", "trip_distance", "pickup_hour", "payment_type"]].reset_index(drop=True),
        reduced_df
    ],
    axis=1
)

display(result_df.head())

=== TRUNCATED SVD RESULT ===
Original matrix shape : (2000000, 623)
Reduced matrix shape  : (2000000, 20)
Explained variance ratio (first components):
[0.20781358 0.01087335 0.0748354  0.06386212 0.05412155 0.05386285
 0.05269996 0.04417966 0.03998079 0.03066967 0.02133727 0.01787459
 0.0107731  0.00842681 0.00812398 0.00799221 0.00794154 0.00787857
 0.00709256 0.00683568]
Total explained variance: 0.7371752463285116


,total_amount,trip_distance,pickup_hour,payment_type,svd_component_1,svd_component_2,svd_component_3,svd_component_4,svd_component_5,svd_component_6,...,svd_component_11,svd_component_12,svd_component_13,svd_component_14,svd_component_15,svd_component_16,svd_component_17,svd_component_18,svd_component_19,svd_component_20
0,95.88,18.25,6,1,6.610912,3.035823,-1.147367,-0.055970,0.582452,-0.589786,...,1.317126,-1.024889,0.532157,-0.244707,0.614990,-0.309842,-0.553780,-0.053059,-0.507071,0.163974
1,20.00,1.80,11,1,-0.292150,1.091018,-2.431826,1.222207,-0.423359,0.286390,...,0.371149,-2.032519,0.802230,0.138117,0.337727,0.619089,-0.204175,0.075556,0.510673,-0.534544
2,13.03,1.24,0,0,-1.759653,1.319844,-0.473253,-0.326780,0.046487,0.033761,...,-0.008626,0.147636,0.288208,-0.131860,-0.380311,-0.628832,0.459794,-0.466889,-0.013592,-0.208093
3,13.12,0.69,11,1,-1.626520,1.756288,-0.251551,-0.278347,-0.069488,0.109520,...,0.506524,0.174669,0.029882,-0.232970,-0.552908,0.465078,-0.442020,-0.120653,-0.294400,0.080787
4,21.28,2.70,20,1,-0.797380,1.647437,0.784007,0.156127,-0.259518,-0.108220,...,-0.376312,-0.311568,0.437576,-0.175860,-0.522339,0.490133,-0.491804,-0.028740,-0.335352,0.104366


## CELL 12 — Xuất kết quả cuối của Task

Kết quả xuất ra gồm:
- **Parquet/CSV đầu vào cho SVD**
- **Parquet/CSV ma trận đã giảm chiều**
- báo cáo explained variance

In [12]:
import json

v7_reduced_parquet = str(V7_DIR / "v7_reduced_svd.parquet")
v7_reduced_csv = str(V7_DIR / "v7_reduced_svd.csv")
svd_report_json = str(REPORT_DIR / "svd_report.json")

# Lưu kết quả giảm chiều
result_df.to_parquet(v7_reduced_parquet, index=False)
result_df.to_csv(v7_reduced_csv, index=False)

# Lưu báo cáo SVD
svd_report = {
    "original_rows": int(X.shape[0]),
    "original_features": int(X.shape[1]),
    "reduced_features": int(X_reduced.shape[1]),
    "n_components": int(N_COMPONENTS),
    "explained_variance_ratio": explained_variance_ratio.tolist(),
    "total_explained_variance": float(explained_variance_sum),
}

with open(svd_report_json, "w", encoding="utf-8") as f:
    json.dump(svd_report, f, ensure_ascii=False, indent=2)

print("Saved reduced parquet:", v7_reduced_parquet)
print("Saved reduced csv    :", v7_reduced_csv)
print("Saved SVD report     :", svd_report_json)

Saved reduced parquet: /content/project_bigdata_svd/data/v7_svd_output/v7_reduced_svd.parquet
Saved reduced csv    : /content/project_bigdata_svd/data/v7_svd_output/v7_reduced_svd.csv
Saved SVD report     : /content/project_bigdata_svd/reports/svd_report.json


## CELL 13 — Tải file kết quả về máy (Google Colab)

Chạy cell này sau khi hoàn tất toàn bộ notebook.

In [13]:
from google.colab import files

files.download(task1_parquet)
files.download(v7_reduced_parquet)
files.download(svd_report_json)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>